# 05 Deep-Dive Analysis

Extends `04_delay_analysis.ipynb`. Notebook 04 describes *what* the delays look like; this notebook asks *why*, and how much of that can be trusted.

> **Synthetic data.** All records are simulated and the findings are illustrative only. Trains and stations are shown by ID.

| Section | Question |
| --- | --- |
| 2 | What does a disruption actually look like? |
| 3 | How much of the network's delay do disruptions explain? |
| 4 | Is the disruption rate different by train or weekday? |
| 5 | Do the train and weekday rankings survive once disruptions are removed? |
| 6 | Does delay carry over from one stop to the next? |
| 7 | How sensitive is on-time performance to the 5 minute threshold? |
| 8 | Dwell, running time and the early-arrival floor |
| 9 | Summary of statistical tests |
| 10 | Key findings and limits |

Read-only. Self-contained: reads `railpulse.duckdb` built by notebooks 01 and 03.

## 1. Setup

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

DB_PATH = next(p / "railpulse.duckdb" for p in [Path.cwd(), *Path.cwd().parents] if (p / "railpulse.duckdb").is_file())
ON_TIME_THRESHOLD_MIN = 5
RNG = np.random.default_rng(42)

con = duckdb.connect(str(DB_PATH), read_only=True)
for table in ["train_run_events_clean", "train_master_clean", "station_master_clean"]:
    assert con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0] > 0, f"{table} is empty, run notebooks 01 and 03"


def query(sql):
    return con.sql(sql).df()


pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

BLUE, ORANGE, GREY = "#2f6db5", "#d9822b", "#8a8f98"


def style(fig, title, subtitle, height=420):
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><sup>{subtitle}</sup>", x=0),
        height=height, template="plotly_white", font=dict(size=13),
        margin=dict(t=90, l=60, r=30, b=60),
    )
    return fig


print(f"Connected to {DB_PATH.name} (read-only)")

Connected to railpulse.duckdb (read-only)


## 2. Anatomy of a disruption

Notebook 04 treats a disruption as a flag on a stop. Here the flags are grouped into episodes, one per run, to see how they are structured.

In [2]:
episodes = query("""
    WITH flagged AS (
        SELECT
            service_id,
            MIN(station_sequence) FILTER (WHERE simulated_disruption_flag) AS first_stop,
            MAX(station_sequence) FILTER (WHERE simulated_disruption_flag) AS last_stop,
            COUNT(*) FILTER (WHERE simulated_disruption_flag) AS disrupted_stops,
            COUNT(DISTINCT simulated_disruption_type) FILTER (WHERE simulated_disruption_flag) AS types_in_run,
            MAX(station_sequence) AS stops_in_run
        FROM train_run_events_clean
        GROUP BY service_id
    )
    SELECT
        disrupted_stops,
        COUNT(*) AS runs,
        MIN(first_stop) AS earliest_first_stop,
        MAX(first_stop) AS latest_first_stop,
        BOOL_AND(last_stop - first_stop + 1 = disrupted_stops) AS always_contiguous,
        COUNT(*) FILTER (WHERE last_stop = stops_in_run) AS ends_at_terminus,
        MAX(types_in_run) AS max_types_in_run
    FROM flagged
    WHERE disrupted_stops > 0
    GROUP BY disrupted_stops
    ORDER BY disrupted_stops
""")
episodes

,disrupted_stops,runs,earliest_first_stop,latest_first_stop,always_contiguous,ends_at_terminus,max_types_in_run
0,4,9,2,2,True,9,1
1,5,14,2,2,True,14,1
2,6,5,2,2,True,5,1


Every disrupted run follows the same template: **one episode, starting at the second stop, lasting 4 to 6 consecutive stops, running to the end of the route, with a single cause.** There are no mid-route disruptions and no recoveries inside an episode. This is a property of how the data was generated, so it should be read as a simulator artefact and not as a pattern about railways.

In [3]:
profile = query("""
    WITH rel AS (
        SELECT
            *,
            station_sequence - MIN(station_sequence) FILTER (WHERE simulated_disruption_flag) OVER (PARTITION BY service_id) AS stops_since_onset
        FROM train_run_events_clean
    )
    SELECT
        stops_since_onset,
        COUNT(arrival_delay_min) AS arrivals,
        ROUND(AVG(arrival_delay_min), 2) AS avg_arrival_delay,
        ROUND(AVG(departure_delay_min), 2) AS avg_departure_delay
    FROM rel
    WHERE stops_since_onset BETWEEN -1 AND 5
    GROUP BY stops_since_onset
    ORDER BY stops_since_onset
""")
profile

,stops_since_onset,arrivals,avg_arrival_delay,avg_departure_delay
0,-1,0,NaN,3.46
1,0,28,6.36,8.04
2,1,28,11.07,13.68
3,2,28,14.57,15.79
4,3,28,16.71,19.16
5,4,19,18.84,15.40
6,5,5,12.20,NaN


In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=profile["stops_since_onset"], y=profile["avg_arrival_delay"], name="Arrival",
                         mode="lines+markers", line=dict(color=BLUE, width=3)))
fig.add_trace(go.Scatter(x=profile["stops_since_onset"], y=profile["avg_departure_delay"], name="Departure",
                         mode="lines+markers", line=dict(color=ORANGE, width=3)))
fig.update_xaxes(title_text="Stops since disruption onset (0 = first disrupted stop)", dtick=1)
fig.update_yaxes(title_text="Average delay (minutes)", rangemode="tozero")
style(fig, "Delay compounds once a disruption starts",
      "Average delay by stop relative to onset, 28 disrupted runs. No recovery before the end of the route. Synthetic data.").show()

## 3. How much of the delay do disruptions explain?

Run-level view: one row per run, comparing the delay at the final stop for disrupted and undisrupted runs.

In [5]:
run_outcome = query(f"""
    WITH runs AS (
        SELECT
            service_id,
            train_id,
            service_date,
            BOOL_OR(simulated_disruption_flag) AS disrupted,
            ARG_MAX(arrival_delay_min, station_sequence) AS final_arrival_delay,
            ROUND(AVG(arrival_delay_min), 2) AS avg_arrival_delay
        FROM train_run_events_clean
        GROUP BY service_id, train_id, service_date
    )
    SELECT * FROM runs
""")

summary = run_outcome.groupby("disrupted").agg(
    runs=("service_id", "count"),
    mean_final_delay=("final_arrival_delay", "mean"),
    median_final_delay=("final_arrival_delay", "median"),
    max_final_delay=("final_arrival_delay", "max"),
    final_on_time_pct=("final_arrival_delay", lambda s: 100 * (s <= ON_TIME_THRESHOLD_MIN).mean()),
).round(2)
summary

,runs,mean_final_delay,median_final_delay,max_final_delay,final_on_time_pct
disrupted,,,,,
False,398,1.87,2.0,13,84.67
True,28,17.61,16.0,32,3.57


In [6]:
# counterfactual: network arrival figures if no run had been disrupted
arrivals = query("SELECT simulated_disruption_flag AS disrupted, arrival_delay_min AS delay FROM train_run_events_clean WHERE arrival_delay_min IS NOT NULL")
arrivals["on_time"] = arrivals["delay"] <= ON_TIME_THRESHOLD_MIN
late = arrivals[~arrivals["on_time"]]

contrib = pd.DataFrame({
    "all arrivals": [len(arrivals), arrivals["delay"].mean(), 100 * arrivals["on_time"].mean(), arrivals["delay"].sum()],
    "undisrupted only": [(~arrivals["disrupted"]).sum(), arrivals.loc[~arrivals["disrupted"], "delay"].mean(),
                         100 * arrivals.loc[~arrivals["disrupted"], "on_time"].mean(), arrivals.loc[~arrivals["disrupted"], "delay"].sum()],
}, index=["observations", "avg_delay_min", "on_time_pct", "total_delay_min"]).round(2)
print(contrib)

excess = arrivals.loc[arrivals["disrupted"], "delay"].sum() - arrivals["disrupted"].sum() * arrivals.loc[~arrivals["disrupted"], "delay"].mean()
print(f"\nDisrupted stops are {100 * arrivals['disrupted'].mean():.1f}% of arrivals but carry {100 * arrivals.loc[arrivals['disrupted'], 'delay'].sum() / arrivals['delay'].sum():.1f}% of all delay minutes.")
print(f"They account for {100 * late['disrupted'].mean():.1f}% of late arrivals; {100 * (1 - late['disrupted'].mean()):.1f}% of late arrivals are ordinary.")
print(f"Excess delay over the undisrupted average: {excess:.0f} min ({100 * excess / arrivals['delay'].sum():.1f}% of the network total).")

                 all arrivals  undisrupted only
observations           2034.0           1898.00
avg_delay_min             2.7              1.95
on_time_pct              79.7             84.25
total_delay_min        5491.0           3708.00

Disrupted stops are 6.7% of arrivals but carry 32.5% of all delay minutes.
They account for 27.6% of late arrivals; 72.4% of late arrivals are ordinary.
Excess delay over the undisrupted average: 1517 min (27.6% of the network total).


In [7]:
bins = [-10, 5, 10, 15, 100]
labels = ["On time (<=5)", "6-10 min", "11-15 min", "Over 15 min"]
arrivals["band"] = pd.cut(arrivals["delay"], bins=bins, labels=labels)
mix = arrivals.groupby(["band", "disrupted"], observed=True).size().unstack(fill_value=0).rename(columns={False: "Undisrupted", True: "Disrupted"})

fig = go.Figure()
for col, colour in [("Undisrupted", BLUE), ("Disrupted", ORANGE)]:
    fig.add_trace(go.Bar(x=mix.index.astype(str), y=mix[col], name=col, marker_color=colour,
                         hovertemplate="%{x}<br>%{y} arrivals<extra>" + col + "</extra>"))
fig.update_layout(barmode="stack")
fig.update_yaxes(title_text="Arrivals", type="log")
style(fig, "Long delays are mostly disrupted; moderate delays mostly are not",
      "Arrivals by delay band, split by disruption flag. Log scale. Synthetic data.").show()
mix

disrupted,Undisrupted,Disrupted
band,,
On time (<=5),1599,22
6-10 min,270,31
11-15 min,29,39
Over 15 min,0,44


Disruptions turn a run from a reliable one into a near-certain failure: undisrupted runs finish on time about 85% of the time, disrupted runs about 4%. Every arrival more than 15 minutes late is disrupted. Disrupted stops are 6.7% of arrivals but carry about a third of all delay minutes, and removing them lifts network on-time performance from 79.7% to 84.3%. Yet disruptions are a minority of all lateness; the 6 to 10 minute band is made up almost entirely of ordinary delays.

## 4. Is the disruption rate different by train or weekday?

Notebook 04 ranks trains by average delay. Part of that ranking is driven by how many disrupted runs each train happened to have. With only 28 disrupted runs, chance alone can reorder the table. A chi-squared test checks whether disruption rates really differ.

In [8]:
runs_by_train = run_outcome.groupby("train_id").agg(runs=("service_id", "count"), disrupted_runs=("disrupted", "sum"))
runs_by_train["disrupted_pct"] = (100 * runs_by_train["disrupted_runs"] / runs_by_train["runs"]).round(1)

chi2, p_train, dof, _ = stats.chi2_contingency(np.array([runs_by_train["disrupted_runs"], runs_by_train["runs"] - runs_by_train["disrupted_runs"]]))
print(runs_by_train.T.to_string())
print(f"\nChi-squared across trains: chi2 = {chi2:.2f}, dof = {dof}, p = {p_train:.3f}")

run_outcome["weekday_no"] = pd.to_datetime(run_outcome["service_date"]).dt.dayofweek
run_outcome["weekday"] = pd.to_datetime(run_outcome["service_date"]).dt.day_name()
runs_by_day = run_outcome.groupby(["weekday_no", "weekday"]).agg(runs=("service_id", "count"), disrupted_runs=("disrupted", "sum")).reset_index()
runs_by_day["disrupted_pct"] = (100 * runs_by_day["disrupted_runs"] / runs_by_day["runs"]).round(1)
chi2_d, p_day, dof_d, _ = stats.chi2_contingency(np.array([runs_by_day["disrupted_runs"], runs_by_day["runs"] - runs_by_day["disrupted_runs"]]))
print()
print(runs_by_day.set_index("weekday")[["runs", "disrupted_runs", "disrupted_pct"]].T.to_string())
print(f"\nChi-squared across weekdays: chi2 = {chi2_d:.2f}, dof = {dof_d}, p = {p_day:.3f}")

train_id        TR001  TR002  TR003  TR004  TR005  TR006  TR007  TR008  TR009  TR010
runs             39.0   45.0   45.0   45.0   39.0   39.0   39.0   45.0   45.0   45.0
disrupted_runs    2.0    3.0    2.0    1.0    2.0    3.0    4.0    4.0    4.0    3.0
disrupted_pct     5.1    6.7    4.4    2.2    5.1    7.7   10.3    8.9    8.9    6.7

Chi-squared across trains: chi2 = 3.71, dof = 9, p = 0.929

weekday         Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
runs              70.0     48.0       60.0      60.0    48.0      70.0    70.0
disrupted_runs     6.0      3.0        2.0       5.0     4.0       4.0     4.0
disrupted_pct      8.6      6.2        3.3       8.3     8.3       5.7     5.7

Chi-squared across weekdays: chi2 = 2.20, dof = 6, p = 0.900


In [9]:
by_type = query("""
    SELECT
        simulated_disruption_type AS disruption_type,
        COUNT(DISTINCT service_id) AS runs,
        COUNT(*) AS stops,
        ROUND(AVG(arrival_delay_min), 2) AS avg_arrival_delay,
        ROUND(AVG(departure_delay_min), 2) AS avg_departure_delay,
        MAX(arrival_delay_min) AS max_arrival_delay
    FROM train_run_events_clean
    WHERE simulated_disruption_flag
    GROUP BY simulated_disruption_type
    ORDER BY avg_arrival_delay DESC
""")
groups = [g["delay"].values for _, g in query("SELECT simulated_disruption_type AS t, arrival_delay_min AS delay FROM train_run_events_clean WHERE simulated_disruption_flag AND arrival_delay_min IS NOT NULL").groupby("t")]
h, p_type = stats.kruskal(*groups)
print(by_type.to_string(index=False))
print(f"\nKruskal-Wallis across disruption types: H = {h:.2f}, p = {p_type:.3f}")

disruption_type  runs  stops  avg_arrival_delay  avg_departure_delay  max_arrival_delay
          track     6     30              15.40                16.04                 33
    operational     6     28              12.86                12.68                 31
         signal     9     43              12.56                13.88                 30
        weather     7     35              12.03                12.68                 27

Kruskal-Wallis across disruption types: H = 2.34, p = 0.505


Disruption rates do **not** differ detectably by train or by weekday (both p > 0.9), so which trains rank worst in notebook 04 is largely luck in where the 28 disrupted runs fell. The disruption types may differ in severity, but the per-type samples are 6 to 9 runs each and the Kruskal-Wallis test finds no detectable difference between types (p about 0.5), so any ordering of causes is weak evidence.

## 5. Do the rankings survive without disruptions?

Rank trains and weekdays again using undisrupted stops only. This isolates the baseline behaviour of each train and day from the random placement of disruptions.

In [10]:
base = query("""
    SELECT service_id, train_id, service_date, station_sequence, arrival_delay_min AS delay
    FROM train_run_events_clean
    WHERE arrival_delay_min IS NOT NULL AND NOT simulated_disruption_flag
""")
base["weekday_no"] = pd.to_datetime(base["service_date"]).dt.dayofweek

all_arr = query("SELECT train_id, arrival_delay_min AS delay FROM train_run_events_clean WHERE arrival_delay_min IS NOT NULL")
rank_all = all_arr.groupby("train_id")["delay"].mean().rank(ascending=False, method="min").astype(int)

# bootstrap by run (stops within a run are not independent)
def bootstrap_ci(frame, n_boot=2000):
    runs = frame.groupby("service_id")["delay"].agg(["sum", "count"])
    s, c = runs["sum"].to_numpy(), runs["count"].to_numpy()
    idx = RNG.integers(0, len(runs), (n_boot, len(runs)))
    means = s[idx].sum(1) / c[idx].sum(1)
    return np.percentile(means, [2.5, 97.5])

rows = []
for train_id, part in base.groupby("train_id"):
    lo, hi = bootstrap_ci(part)
    rows.append(dict(train_id=train_id, arrivals=len(part), avg_delay=part["delay"].mean(), ci_low=lo, ci_high=hi))
by_train_base = pd.DataFrame(rows).set_index("train_id")
by_train_base["rank_undisrupted"] = by_train_base["avg_delay"].rank(ascending=False, method="min").astype(int)
by_train_base["rank_all_stops"] = rank_all
by_train_base["rank_shift"] = by_train_base["rank_all_stops"] - by_train_base["rank_undisrupted"]
by_train_base = by_train_base.sort_values("avg_delay", ascending=False).round(2)
print(by_train_base.to_string())

h, p = stats.kruskal(*[g["delay"].values for _, g in base.groupby("train_id")])
print(f"\nKruskal-Wallis across trains, undisrupted stops: H = {h:.2f}, p = {p:.4f}")
rho, p_rho = stats.spearmanr(by_train_base["rank_all_stops"], by_train_base["rank_undisrupted"])
print(f"Rank agreement with notebook 04 ranking: Spearman rho = {rho:.2f} (p = {p_rho:.3f})")

          arrivals  avg_delay  ci_low  ci_high  rank_undisrupted  rank_all_stops  rank_shift
train_id                                                                                    
TR005          222       2.44    1.74     3.18                 1               3           2
TR009          205       2.38    1.40     3.35                 2               2           0
TR001          148       2.34    1.59     3.13                 3               6           3
TR006          216       2.17    1.20     3.19                 4               4           0
TR007          175       2.06    1.42     2.72                 5               1          -4
TR010          210       1.94    1.23     2.75                 6               7           1
TR008          164       1.82    1.12     2.53                 7               5          -2
TR004          176       1.77    0.88     2.71                 8               9           1
TR002          210       1.36    0.91     1.83                 9      

In [11]:
ordered = by_train_base.sort_values("avg_delay")
fig = go.Figure(go.Scatter(
    y=ordered.index, x=ordered["avg_delay"], mode="markers", marker=dict(color=BLUE, size=11),
    error_x=dict(type="data", symmetric=False, array=ordered["ci_high"] - ordered["avg_delay"],
                 arrayminus=ordered["avg_delay"] - ordered["ci_low"], color=GREY, thickness=2),
    hovertemplate="%{y}<br>Average %{x:.2f} min<extra></extra>",
))
network_base = base["delay"].mean()
fig.add_vline(x=network_base, line=dict(color=GREY, dash="dot"))
fig.update_xaxes(title_text="Average arrival delay, undisrupted stops (minutes)")
style(fig, "Train differences are small and most intervals overlap",
      f"Mean with 95% bootstrap interval resampled by run. Dotted line = undisrupted network mean {network_base:.2f} min. Synthetic data.", height=460).show()

In [12]:
by_day_base = base.groupby("weekday_no").agg(arrivals=("delay", "size"), avg_delay=("delay", "mean"), dates=("service_date", "nunique"))
by_day_base.insert(0, "weekday", ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
by_day_base["avg_delay"] = by_day_base["avg_delay"].round(2)
h, p = stats.kruskal(*[g["delay"].values for _, g in base.groupby("weekday_no")])
print(by_day_base.to_string(index=False))
print(f"\nKruskal-Wallis across weekdays, undisrupted stops: H = {h:.2f}, p = {p:.4f}")

  weekday  arrivals  avg_delay  dates
   Monday       307       1.73      7
  Tuesday       207       1.53      6
Wednesday       279       1.53      6
 Thursday       266       2.30      6
   Friday       208       2.82      6
 Saturday       316       1.97      7
   Sunday       315       1.95      7

Kruskal-Wallis across weekdays, undisrupted stops: H = 20.88, p = 0.0019


Removing disruptions changes the story. TR007, ranked worst in notebook 04, is only mid-table once its four disrupted runs are set aside. A real baseline difference between trains remains (p about 0.002), but it is small, about 1.3 minutes between best (TR003) and worst (TR005), and the intervals overlap heavily. The two rankings still agree moderately (Spearman 0.78), so notebook 04's order is not wrong, only less certain than it looks. Friday stays the weakest weekday on undisrupted stops, so that finding in notebook 04 holds up better than the train ranking does.

## 6. Does delay carry over?

Notebook 04 showed delay grows while standing and shrinks while running. Here: does a late start predict a late finish?

In [13]:
carry = query("""
    SELECT
        service_id,
        BOOL_OR(simulated_disruption_flag) AS disrupted,
        MAX(departure_delay_min) FILTER (WHERE is_origin) AS origin_departure_delay,
        ARG_MAX(arrival_delay_min, station_sequence) AS final_arrival_delay
    FROM train_run_events_clean
    GROUP BY service_id
""")
rows = []
for label, part in [("Undisrupted runs", carry[~carry["disrupted"]]), ("Disrupted runs", carry[carry["disrupted"]])]:
    r, p = stats.spearmanr(part["origin_departure_delay"], part["final_arrival_delay"])
    rows.append(dict(group=label, runs=len(part), spearman_rho=round(r, 3), p_value=round(p, 3)))
pd.DataFrame(rows)

,group,runs,spearman_rho,p_value
0,Undisrupted runs,398,-0.006,0.905
1,Disrupted runs,28,0.296,0.127


In [14]:
# stop-to-stop: arrival delay at stop k vs departure delay at stop k-1, and arrival vs next arrival
steps = query("""
    SELECT
        DISTINCT a.service_id,
        a.station_sequence,
        a.simulated_disruption_flag AS disrupted,
        a.departure_delay_min AS prev_departure,
        b.arrival_delay_min AS next_arrival
    FROM train_run_events_clean a
    JOIN train_run_events_clean b ON b.service_id = a.service_id AND b.station_sequence = a.station_sequence + 1
    WHERE a.departure_delay_min IS NOT NULL AND b.arrival_delay_min IS NOT NULL
""")
for label, part in [("Undisrupted", steps[~steps["disrupted"]]), ("Disrupted", steps[steps["disrupted"]])]:
    r = part["prev_departure"].corr(part["next_arrival"])
    slope = np.polyfit(part["prev_departure"], part["next_arrival"], 1)[0]
    print(f"{label:12s} stop pairs: {len(part):5d}  correlation {r:5.2f}  slope {slope:5.2f}")

Undisrupted  stop pairs:  1926  correlation  0.46  slope  0.40
Disrupted    stop pairs:   108  correlation  0.81  slope  0.77


For undisrupted runs a late departure is largely forgotten by the final stop (rank correlation close to zero), which means ordinary delay is **short-lived noise** that is absorbed by schedule slack. Within a single stop pair the carry-over is real but partial, with a slope well below 1. Disruptions are the only mechanism that makes delay persist. In practice that means a late origin departure is not by itself a reason to expect a late arrival.

## 7. Sensitivity to the on-time threshold

The 5 minute threshold in notebook 04 is an assumption. How much do the headline figures depend on it?

In [15]:
sens = query("""
    SELECT
        t AS threshold_min,
        ROUND(100.0 * AVG((arrival_delay_min <= t)::INT), 1) AS arrival_on_time_pct,
        ROUND(100.0 * AVG((departure_delay_min <= t)::INT), 1) AS departure_on_time_pct
    FROM train_run_events_clean, UNNEST([0, 1, 3, 5, 10, 15]) AS u(t)
    WHERE arrival_delay_min IS NOT NULL
    GROUP BY t
    ORDER BY t
""")
sens

,threshold_min,arrival_on_time_pct,departure_on_time_pct
0,0,37.6,25.2
1,1,47.0,33.5
2,3,64.4,49.3
3,5,79.7,67.0
4,10,94.5,90.9
5,15,97.8,97.0


In [16]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=sens["threshold_min"], y=sens["arrival_on_time_pct"], name="Arrival", mode="lines+markers",
                         line=dict(color=BLUE, width=3)))
fig.add_trace(go.Scatter(x=sens["threshold_min"], y=sens["departure_on_time_pct"], name="Departure", mode="lines+markers",
                         line=dict(color=ORANGE, width=3)))
fig.add_vline(x=ON_TIME_THRESHOLD_MIN, line=dict(color=GREY, dash="dot"))
fig.update_xaxes(title_text="On-time threshold (minutes late)", dtick=1)
fig.update_yaxes(title_text="On time (%)", range=[0, 100])
style(fig, "On-time performance depends heavily on the threshold",
      "Share of observations within the threshold. Dotted line = threshold used in notebook 04. Synthetic data.").show()

In [17]:
# uncertainty around the headline figure, resampled by run
per_run = query("""
    SELECT service_id, COUNT(arrival_delay_min) AS n, SUM((arrival_delay_min <= 5)::INT) AS on_time
    FROM train_run_events_clean WHERE arrival_delay_min IS NOT NULL GROUP BY service_id
""")
idx = RNG.integers(0, len(per_run), (5000, len(per_run)))
boot = 100 * per_run["on_time"].to_numpy()[idx].sum(1) / per_run["n"].to_numpy()[idx].sum(1)
point = 100 * per_run["on_time"].sum() / per_run["n"].sum()
print(f"Network arrival on-time at 5 min: {point:.1f}%  (95% bootstrap interval {np.percentile(boot, 2.5):.1f}% to {np.percentile(boot, 97.5):.1f}%)")

Network arrival on-time at 5 min: 79.7%  (95% bootstrap interval 76.9% to 82.5%)


Moving the threshold from 3 to 10 minutes swings arrival on-time performance from about 64% to 94%. A target or a ranking built on this metric should state its threshold. The 5 minute headline carries a plausible range of roughly 77% to 82% from sampling alone.

## 8. Dwell time, running time and the early-arrival floor

In [18]:
dwell = query("""
    SELECT
        simulated_disruption_flag AS disrupted,
        COUNT(*) AS stops,
        ROUND(AVG(scheduled_dwell_min), 2) AS scheduled_dwell,
        ROUND(AVG(actual_dwell_min), 2) AS actual_dwell,
        ROUND(AVG(actual_dwell_min - scheduled_dwell_min), 2) AS overrun,
        ROUND(100.0 * AVG((actual_dwell_min > scheduled_dwell_min)::INT), 1) AS pct_over_schedule,
        ROUND(CORR(actual_dwell_min - scheduled_dwell_min, arrival_delay_min), 3) AS overrun_vs_arrival_delay
    FROM train_run_events_clean
    WHERE actual_dwell_min IS NOT NULL
    GROUP BY simulated_disruption_flag
    ORDER BY disrupted
""")
dwell

,disrupted,stops,scheduled_dwell,actual_dwell,overrun,pct_over_schedule,overrun_vs_arrival_delay
0,False,1488,4.52,6.05,1.53,63.4,-0.010
1,True,107,4.54,6.46,1.92,73.8,0.044


In [19]:
segments = query("""
    WITH legs AS (
        SELECT
            LAG(station_id) OVER run AS from_station,
            station_id AS to_station,
            simulated_disruption_flag AS disrupted,
            date_diff('minute', LAG(scheduled_departure) OVER run, scheduled_arrival) AS scheduled_run_min,
            date_diff('minute', LAG(actual_departure) OVER run, actual_arrival) AS actual_run_min
        FROM train_run_events_clean
        WINDOW run AS (PARTITION BY service_id ORDER BY station_sequence)
    )
    SELECT
        COUNT(*) AS legs,
        ROUND(AVG(scheduled_run_min), 2) AS avg_scheduled_run_min,
        ROUND(AVG(actual_run_min), 2) AS avg_actual_run_min,
        ROUND(AVG(actual_run_min - scheduled_run_min), 2) AS avg_gain_min,
        ROUND(100.0 * AVG((actual_run_min < scheduled_run_min)::INT), 1) AS pct_faster_than_schedule,
        ROUND(100.0 * AVG((actual_run_min > scheduled_run_min)::INT), 1) AS pct_slower_than_schedule
    FROM legs
    WHERE scheduled_run_min IS NOT NULL AND actual_run_min IS NOT NULL AND NOT disrupted
""")
segments

,legs,avg_scheduled_run_min,avg_actual_run_min,avg_gain_min,pct_faster_than_schedule,pct_slower_than_schedule
0,1898,68.92,67.37,-1.55,60.6,30.0


In [20]:
early = query("""
    SELECT
        COUNT(*) AS arrivals,
        ROUND(100.0 * AVG((arrival_delay_min < 0)::INT), 1) AS pct_early,
        ROUND(100.0 * AVG((arrival_delay_min = -2)::INT), 1) AS pct_at_floor,
        ROUND(AVG(arrival_delay_min), 2) AS avg_delay_as_stored,
        ROUND(AVG(arrival_late_min), 2) AS avg_lateness_floor_at_zero
    FROM train_run_events_clean
    WHERE arrival_delay_min IS NOT NULL
""")
early

,arrivals,pct_early,pct_at_floor,avg_delay_as_stored,avg_lateness_floor_at_zero
0,2034,28.8,20.3,2.7,3.19


Three observations:

- Trains overrun their scheduled dwell by about 1.5 minutes on average, and 64% of stops overrun. This is the mechanism behind notebook 04's finding that delay is added at stops. It is almost uncorrelated with arrival delay, so dwell overrun is not what pushes a train past the on-time threshold.
- Undisrupted trains cover each leg faster than scheduled by about 1.5 minutes. The timetable carries running-time padding that is consumed by the dwell overruns, which is why ordinary delay does not accumulate along the run (section 6).
- About 29% of arrivals are early and 20% sit exactly at the -2 minute floor. The floor is a data-generation limit, so the true spread of early running is hidden and the signed average delay is biased upward. Lateness with early running counted as zero (`arrival_late_min`) is a cleaner measure for reporting.

## 9. Statistical tests at a glance

In [21]:
def kw(frame, group_col):
    return stats.kruskal(*[g["delay"].values for _, g in frame.groupby(group_col)])

all_arrivals = query("SELECT train_id, service_date, simulated_disruption_flag AS disrupted, arrival_delay_min AS delay FROM train_run_events_clean WHERE arrival_delay_min IS NOT NULL")
all_arrivals["weekday_no"] = pd.to_datetime(all_arrivals["service_date"]).dt.dayofweek
und = all_arrivals[~all_arrivals["disrupted"]]

u, p_u = stats.mannwhitneyu(all_arrivals.loc[all_arrivals["disrupted"], "delay"], all_arrivals.loc[~all_arrivals["disrupted"], "delay"])
tests = pd.DataFrame([
    ("Disrupted vs undisrupted arrival delay", "Mann-Whitney U", p_u, "Very strong difference"),
    ("Disruption rate by train (runs)", "Chi-squared", p_train, "No detectable difference"),
    ("Disruption rate by weekday (runs)", "Chi-squared", p_day, "No detectable difference"),
    ("Arrival delay by train, all stops", "Kruskal-Wallis", kw(all_arrivals, "train_id").pvalue, "Differences present"),
    ("Arrival delay by train, undisrupted", "Kruskal-Wallis", kw(und, "train_id").pvalue, "Differences present, small"),
    ("Arrival delay by weekday, all stops", "Kruskal-Wallis", kw(all_arrivals, "weekday_no").pvalue, "Differences present"),
    ("Arrival delay by weekday, undisrupted", "Kruskal-Wallis", kw(und, "weekday_no").pvalue, "Differences present, small"),
], columns=["comparison", "test", "p_value", "reading"])
tests["p_value"] = tests["p_value"].map(lambda p: f"{p:.2g}")
tests

,comparison,test,p_value,reading
0,Disrupted vs undisrupted arrival delay,Mann-Whitney U,7.5e-65,Very strong difference
1,Disruption rate by train (runs),Chi-squared,0.93,No detectable difference
2,Disruption rate by weekday (runs),Chi-squared,0.9,No detectable difference
3,"Arrival delay by train, all stops",Kruskal-Wallis,0.00093,Differences present
4,"Arrival delay by train, undisrupted",Kruskal-Wallis,0.0023,"Differences present, small"
5,"Arrival delay by weekday, all stops",Kruskal-Wallis,0.00027,Differences present
6,"Arrival delay by weekday, undisrupted",Kruskal-Wallis,0.0019,"Differences present, small"


Non-parametric tests are used because delays are integer minutes with a long right tail. Stops within a run are not independent, so the p-values for stop-level tests are optimistic; the bootstrap intervals above resample whole runs to account for that. With seven tests, the borderline results should not be over-read.

## 10. Key findings

Synthetic data: illustrative, not evidence about any real service.

| # | Finding | Evidence |
| --- | --- | --- |
| 1 | A disruption is a run-level event, not a stop-level one | 28 of 426 runs (6.6%); always one episode of 4 to 6 stops from stop 2 to the end, one cause |
| 2 | Disruptions dominate the severe tail but not the total | Every arrival over 15 min is disrupted, yet 72% of late arrivals are ordinary; removing disruptions lifts on-time from 79.7% to 84.3% |
| 3 | A disrupted run almost never finishes on time | 3.6% against 84.7% on time at the last stop; mean final delay 17.6 against 1.9 min |
| 4 | Disruption frequency is not tied to a train or weekday | Chi-squared p = 0.93 and p = 0.90 |
| 5 | Notebook 04's train ranking is less certain than it looks | TR007 falls from 1st to 5th without its 4 disrupted runs; rankings agree at Spearman 0.78; most intervals overlap |
| 6 | A small baseline difference between trains and weekdays is real | Undisrupted Kruskal-Wallis p about 0.002; Friday weakest at 2.8 min against 1.5 on Tuesday and Wednesday |
| 7 | Ordinary delay does not carry across a run | Origin departure delay vs final arrival delay: rank correlation about 0 |
| 8 | The 5 minute threshold matters a great deal | 64% at 3 min, 80% at 5 min, 94% at 10 min; 5 min interval about 77% to 82% |
| 9 | Timetable padding offsets dwell overrun | Dwell +1.5 min per stop; running legs 1.5 min faster than scheduled |

**Recommendations for the reader of notebook 04**

- Report disrupted and undisrupted performance separately; a blended average mixes two different processes.
- Do not act on the train ranking without the intervals in section 5.
- State the on-time threshold next to every on-time percentage.
- Treat the weekday result as indicative: each weekday has only 6 or 7 dates.

**Limits:** 45 days, 10 trains and 28 disrupted runs make type-level and train-level conclusions fragile. The disruption template (start at stop 2, run to the end) is almost certainly an artefact of the simulator. Early arrivals are floored at -2 minutes in the source. No test here adjusts for multiple comparisons.

In [22]:
con.close()
print("Connection closed.")

Connection closed.
